# JSI_wall_heat_flux test case


In [ ]:
from pathlib import Path
import pandas
import numpy as np
from trustutils import run
import sys

run.TRUST_parameters("1.9.6")
run.reset()
run.initBuildDirectory()

## Introduction
 
 Validation made by: jc283033

 Report generated 28/08/2025

### Description
 
The objective is to compare the performance of the Kurul–Podowski wall heat flux model and a simpler bilinear approach. The experimental data are based on a not-yet-published work by the Jožef Stefan Institute. The available data are the wall temperature, the heat flux received by the fluid, and the void volume at four measurement sections.

An efficient wall heat-flux model should be able to predict both the wall temperature from the imposed heat-flux and the correct void volume.

To be used in this case, the Kurul-Podowski departure bubble diameter sub-model had to be slightly modified:

### Original model: 

$$ D_{b} = 10^{-4}(T_l - T_{sat}) + 0.0014 $$

### Modified model:

$$ D_{b} = 10^{-4}(T_l - T_{sat}) + dd $$

The parameter $dd$ has been implemented as a user defined parameter with a base value of 0.0014 m.

The bilinear model is implemented as below:

### Bilinear model:
$$
q =
\begin{cases}
    h_m \cdot (T_p - T_l), & \text{ if },  T_p < T_{sat} \\
    h_d \cdot (T_p - T_{sat}) + \text{offset} , & \text{ else }
\end{cases}
$$
The flux is then splitted between the liquid and evaporation as follows:
- **If** : $T_w < T_{sat}$ everything goes in the liquid 
- **Else If** : $T_l > T_{sat}$ everything goes in the evaporation term
- **Else** :
$$
\begin{aligned}
T_{OSV} &= T_S - c_{OSV} q \\
\epsilon &= \text{clip}\left(\frac{T_l - T_{OSV}}{T_{sat} - T_{OSV}},\; 0, \;1 \right) \\
q_l &= (1 - \epsilon) q \\
q_{l \rightarrow g} &= \epsilon q
\end{aligned}
$$

The parameter $h_m$, $h_d$, $c_{osv}$ are mandatory user defined parameter, $offset$ is also user defined with a base value of 10 $kWm^{-2}$ for inheritance with J. Carrico internship.

# User input

In [ ]:
# Can be changed

number_of_partitions = 1

dparam = {}
dparam["tmax"] = 4
dparam["nb_pas_dt_max"] = 200
dparam["seuil_statio"] = 1e-4
dparam["facsec"] = 0.5
dparam["max_facsec"] = 1

# Shouldn't be changed
dataroot = "jdd"
rho_l = 1333

## Heat flux parameter

In [ ]:
# Heat Flux parameter

# Kurul-Podowski
departure_diameters = [0.05e-3, 0.1e-3] # tested dd parameter in the modified Kurul Podowski model

# Bilinear
dico_bilinear = {}
dico_bilinear[0] = {}
dico_bilinear[0]["hm"] = 1588.1275278711269
dico_bilinear[0]["hd"] = 5596.892769932657
dico_bilinear[0]["off"] = 10000
dico_bilinear[0]["c_osv"] = 0.00025

def dico_kurul(d):
    dico = {}
    dico['dd'] = d
    return dico

## Flow parameter

In [ ]:
dico_flow = { }
dico_flow['Q'] = 600 #surface masse flow kg/m2/s
dico_flow['vi'] = dico_flow['Q']/rho_l #injection refriferent speed m/s
dico_flow['db'] = 0.0003  #buble diameter m
dico_flow['Ti'] = 27 #injection refriferent temperature °C

## Wall initial and limit condition

In [ ]:
def dico_cond(Cl_type):
    dico = {}
    dico['Ci'] = "temperature  Champ_fonc_xyz dom_so 1 201.67406174*z*z*z*z-344.57903334*z*z*z+235.28426812*z*z-87.25263019*z+54.60024946"
    if Cl_type == 'flux' :
        dico['Cl'] = "Neumann_Paroi Champ_front_fonc_xyz 1 (-775.9804814*z*z*z*z+340.81232575*z*z*z+479.01046715*z*z-372.30517073*z+81.31783661)*1000"
    else :
        dico['Cl'] = "Paroi_temperature_imposee Champ_front_fonc_xyz 1 201.67406174*z*z*z*z-344.57903334*z*z*z+235.28426812*z*z-87.25263019*z+54.60024946"
        
    return dico

# Computation launcher

In [ ]:
case_names = []

#Kurl podowski case
for d in departure_diameters:
    for Cl_type in ('flux','temp'):
        dic_cond = dico_cond(Cl_type)
        dic_kurul = dico_kurul(d)
        dd = {**dparam, **dic_kurul, **dico_flow,**dic_cond}
        myrun = run.addCaseFromTemplate(f"{dataroot}_kurul_podowski.data",
                                                    f"{Cl_type}_Kurul_podowski_dd_{d}",
                                                    dd,
                                                    nbProcs=number_of_partitions)
        case_names.append(f"{Cl_type}_Kurul_podowski_dd_{d}")
        if number_of_partitions > 1:
            myrun.partition()

# Bilinear case
for c in range (len(dico_bilinear)):
    for Cl_type in ('flux','temp'):
            dic_cond = dico_cond(Cl_type)
            dd = {**dparam, **dico_bilinear[c], **dico_flow,**dic_cond}
            myrun = run.addCaseFromTemplate(f"{dataroot}_bilineaire.data",
                                                        f"{Cl_type}_Bilinear_case_{c}",
                                                        dd,
                                                        nbProcs=number_of_partitions)
            case_names.append(f"{Cl_type}_Bilinear_case_{c}")
            if number_of_partitions > 1:
                myrun.partition()


run.printCases()

In [ ]:
run.runCases()

## Computer Performance

In [ ]:
run.tablePerf()

# Post processing

In [ ]:

from trustutils import plot, visit

## Experimental Wall temperature

In [ ]:
def Compute_tw_exp (z) :
    return 201.67406174*z*z*z*z-344.57903334*z*z*z+235.28426812*z*z-87.25263019*z+54.60024946
Z = np.linspace(0,0.585,176)

Tw_exp = Compute_tw_exp (Z)

## Wall temperature comparaison

In [ ]:
Graph = plot.Graph("Wall temperature comparison", size=[15, 8])

Graph.add(Z, Tw_exp, marker="-s", label="Experimental data")

for case in case_names :
    dossier = Path(f"{run.BUILD_DIRECTORY}/{case}")

    file = list(dossier.glob("*T_SO.son"))[0]

    Graph.addSegment(f"{case}/{file.name}", marker="-", label=f"{case}")


## Gaz production comparaion

### Gaz Volume in the whole tube

In [ ]:
a = visit.Show(empty=True, title="Total Gaz volume", name="total_gaz")

for case in case_names :
    dossier = Path(f"{run.BUILD_DIRECTORY}/{case}")

    file = list(dossier.glob("*.lata"))[0]
    file = file.name
    
    # for archiving
    run.saveFileAccumulator(f"{case}/{file}")
    
    lata_volume_maille = list(dossier.glob(f"*.lata.VOLUME_MAILLE*"))
    for fff in lata_volume_maille:
        print(f"save {case}/{fff.name}")
        run.saveFileAccumulator(f"{case}/{fff.name}")

    a.visitCommand(f"OpenDatabase('{case}/{file}')")
    a.visitCommand(f'DefineScalarExpression("Gaz_volume_{case}", "ALPHA_1_ELEM_dom_flu*VOLUME_MAILLE_ELEM_dom_flu")')
    a.visitCommand(f'AddPlot("Pseudocolor", "Gaz_volume_{case}")')
    a.visitCommand('DrawPlots()')

    a.visitCommand('QueryOverTimeAtts = GetQueryOverTimeAttributes()')
    a.visitCommand('QueryOverTimeAtts.timeType = QueryOverTimeAtts.DTime')
    a.visitCommand('QueryOverTimeAtts.createWindow = 0')
    a.visitCommand('QueryOverTimeAtts.windowId = 2')
    a.visitCommand('SetQueryOverTimeAttributes(QueryOverTimeAtts)')
    a.visitCommand(f'QueryOverTime("Variable Sum", vars=("Gaz_volume_{case}"), stride=1)')

a.visitCommand('SetActiveWindow(2)')
a.plot()

### Gaz Volume at the observed sections

#### Experimental datas

In [ ]:
Result_exp = [23.65, 36.12, 40.9, 43.39] # mm3

In [ ]:
a = visit.Show(empty=True, title="Empty but necessary", name="wall_temp")

Observed_vol = []
for case in case_names:
    dossier = Path(f"{run.BUILD_DIRECTORY}/{case}")

    file = list(dossier.glob("*.lata"))[0]
    file = file.name

    # for archiving
    run.saveFileAccumulator(f"{case}/{file}")

    a.visitCommand(f"OpenDatabase('{case}/{file}')")
    a.visitCommand("""n = TimeSliderGetNStates()
SetTimeSliderState(n - 1)""")
    a.visitCommand('DefineScalarExpression("Gaz_volume_P1", "ALPHA_1_ELEM_dom_flu*VOLUME_MAILLE_ELEM_dom_flu*lt(coord(ALPHA_1_ELEM_dom_flu)[2],0.05818)*gt(coord(ALPHA_1_ELEM_dom_flu)[2],0)")')
    a.visitCommand('DefineScalarExpression("Gaz_volume_P2", "ALPHA_1_ELEM_dom_flu*VOLUME_MAILLE_ELEM_dom_flu*lt(coord(ALPHA_1_ELEM_dom_flu)[2],0.07818)*gt(coord(ALPHA_1_ELEM_dom_flu)[2],0.020)")')
    a.visitCommand('DefineScalarExpression("Gaz_volume_P3", "ALPHA_1_ELEM_dom_flu*VOLUME_MAILLE_ELEM_dom_flu*lt(coord(ALPHA_1_ELEM_dom_flu)[2],0.09818)*gt(coord(ALPHA_1_ELEM_dom_flu)[2],0.040)")')
    a.visitCommand('DefineScalarExpression("Gaz_volume_P4", "ALPHA_1_ELEM_dom_flu*VOLUME_MAILLE_ELEM_dom_flu*lt(coord(ALPHA_1_ELEM_dom_flu)[2],0.11818)*gt(coord(ALPHA_1_ELEM_dom_flu)[2],0.060)")')

    a.visitCommand("R = []")

    for i in range(1, 5):
        a.visitCommand(f"""
AddPlot("Pseudocolor", "Gaz_volume_P{i}")
DrawPlots()
Query("Variable Sum")
ratio = GetQueryOutputValue()
R.append(ratio*72e9)
DeleteActivePlots()
""")

    # for archiving
    run.saveFileAccumulator(f"temp_R_{case}.txt")

    a.visitCommand(f"""
f = open("temp_R_{case}.txt", "w")
for val in R:
    f.write(str(val) + "\\n")
f.close()
""")
a.plot()

for case in case_names:
    R = []
    with open(f"{run.BUILD_DIRECTORY}/temp_R_{case}.txt") as f:
        for line in f:
            R.append(float(line.strip()))

    Observed_vol.append(R)

Graph = plot.Graph("Wall temperature comparaison", size=[15, 8])
x = [1, 2, 3, 4]

for gg, val in enumerate(Observed_vol):
    Graph.add(x, val, label=f"{case_names[gg]}")
Graph.add(x, Result_exp, marker="-s", label=f"experimental datas", color='black')

# Void fraction and Liquid Temperature field

In [ ]:
for case in case_names :
    dossier = Path(f"{run.BUILD_DIRECTORY}/{case}")
    print(dossier)

    file = list(dossier.glob("*.lata"))[0]
    file = file.name

    # for archiving
    run.saveFileAccumulator(f"{case}/{file}")

    fig = visit.Show(f"{case}/{file}", "Pseudocolor", "ALPHA_1_ELEM_dom_flu", plotmesh=False, title=f"Champs de taux de vide et de température du cas {case} ",nY=2,nX=1)
    fig.slice(origin=[0, 0, 0], normal=[-0.04361938736533601, 0.9990482215818578, 0], type_op='slice2d')
    fig.add(f"{case}/{file}", "Pseudocolor", "TEMPERATURE_0_ELEM_dom_flu", plotmesh=False, xIndice=0, yIndice=1)
    fig.slice(origin=[0, 0, 0], normal=[-0.04361938736533601, 0.9990482215818578, 0], type_op='slice2d')
    fig.plot()

In [ ]:

# for archiving: save all pngs
dossier = Path(run.BUILD_DIRECTORY)

list_png = list(dossier.glob("*.png"))
for f in list_png:
    print(f.name)
    run.saveFileAccumulator(f.name)
